In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score

# Load the dataset
data = pd.read_csv("../COMP30027_2024_asst2_data/train_select.csv")
test_data = pd.read_csv("../COMP30027_2024_asst2_data/test_select.csv")

# Drop the 'id' column as it's just an identifier
data = data.drop(columns=['id'])
test_data = test_data.drop(columns=['id'])

# Split the data into features and target
X = data.drop(columns=['imdb_score_binned'])
y = data['imdb_score_binned']

# Splitting the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

test_data_scaled = scaler.transform(test_data)


In [5]:
# Define the base models
base_models = [
    ('lr', LogisticRegression(random_state=715)),
    ('dt', DecisionTreeClassifier(random_state=715)),
    ('rf', RandomForestClassifier(random_state=715)),
    ('gb', GradientBoostingClassifier(random_state=715))
]

# Meta-learner model
meta_learner = LogisticRegression(random_state=715, max_iter=1000, solver='saga')

# Stacking ensemble model
stacking_model = StackingClassifier(estimators=base_models, final_estimator=meta_learner, cv=5)

# Fit the stacking model on the training data
stacking_model.fit(X_train_scaled, y_train)

C:\Users\Steven\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Steven\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/

StackingClassifier(cv=5,
                   estimators=[('lr', LogisticRegression(random_state=715)),
                               ('dt', DecisionTreeClassifier(random_state=715)),
                               ('rf', RandomForestClassifier(random_state=715)),
                               ('gb',
                                GradientBoostingClassifier(random_state=715))],
                   final_estimator=LogisticRegression(max_iter=1000,
                                                      random_state=715,
                                                      solver='saga'))

In [6]:
# Predict on the test set
y_pred = stacking_model.predict(X_test_scaled)

# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy of the stacking model:", accuracy)

Accuracy of the stacking model: 0.7520798668885191


In [7]:
test_pred = stacking_model.predict(test_data_scaled)

ids = range(1, len(test_pred) + 1)
prediction = pd.DataFrame({'id': ids, 'imdb_score_binned': test_pred})
print(prediction['imdb_score_binned'].value_counts())
prediction.to_csv("../Kaggle_submissions/stacking_model.csv", index=False)

imdb_score_binned
1    522
3    156
0     26
4     25
2     23
Name: count, dtype: int64
